In [1]:
%load_ext autoreload
%autoreload 2

import requests, json, time
import os
import numpy as np
import matplotlib.pyplot as plt
import random as rnd
from tqdm.notebook import tqdm
import re
import pandas as pd
import matplotlib.pyplot as plt
from lxml import etree as et
from collections import Counter
import birth_death_utils as bd
import artificial_collations as ac
import pickle
import warnings
import networkx
warnings.filterwarnings('ignore')

In [2]:
def generate_nexus(sw, dates, name, max_val):
    matrix = ''
    for ms, s in sw.items():
        matrix += ms+'|'+str(dates[ms])+'\t'+s+'\n'

    char_set = ''.join(map(str,range(max_val+1)))

    config_s = f'''#NEXUS

    BEGIN DATA;
        DIMENSIONS NTAX={len(sw.keys())} NCHAR={len(s)};
        FORMAT DATATYPE=STANDARD MISSING=? GAP=-  SYMBOLS="{char_set}";
    MATRIX
    {matrix}
    ;
    END;
    '''

    return config_s

In [3]:
def to_newick(G, root=None, node_attrs=None, edge_attrs=None, branch_attr="weight"):
    if root is None:
        roots = [n for n, d in G.in_degree() if d == 0]
        if len(roots) != 1:
            raise ValueError(f"Expected exactly one root, found {len(roots)}")
        root = roots[0]

    def build(node, parent):
        children = list(G.successors(node))

        # Recurse into children first
        if children:
            subtrees = ",".join(build(c, node) for c in children)
            label = f"({subtrees}){node}"
        else:
            label = f"{node}"

        # Branch length (from the edge connecting parent -> node)
        branch_len = ""
        if parent is not None and branch_attr:
            bl = G.edges[parent, node].get(branch_attr)
            if bl is not None:
                branch_len = f":{bl}"

        # NHX metadata (node attrs + edge attrs, merged)
        nhx_pairs = []
        if node_attrs:
            for a in node_attrs:
                if a in G.nodes[node]:
                    nhx_pairs.append(f"{a}={G.nodes[node][a]}")
        if edge_attrs and parent is not None:
            for a in edge_attrs:
                if a in G.edges[parent, node]:
                    nhx_pairs.append(f"{a}={G.edges[parent, node][a]}")

        nhx = f"[{':'.join(nhx_pairs)}]" if nhx_pairs else ""

        return f"{label}{branch_len}{nhx}"

    return build(root, None) + ";"

In [4]:
### parameter values

lda = 3.2*10**(-2)
mu = 1.4*10**(-2)
Tact = 250
Tinact = 250
n_witnesses = 8
n_verses = 100
p_new_locus = 0.05
p_mod = 0.05
p_inov = 0.2
n_sims = 100

path = 'simulations_data/simulated_traditions_22_07'
os.mkdir(path)

with open(f'{path}/parameters.json', 'w') as f:
    json.dump(
        {'lambda':lda,
        'mu':mu,
        'Tact':Tact,
        'Tinact':Tinact,
        'witnesses_number':n_witnesses,
        'verse_numbers':n_verses,
        'p_new_locus':p_new_locus,
        'p_mod':p_mod,
        'p_inov':p_inov,
        'simulation_numbers':n_sims},
        f
    )

In [8]:
### Simulations

for k in tqdm(range(n_sims)):
    g = bd.generate_tree_fixed_size(lda, mu, Tact, Tinact, n_witnesses)
    inv,seqs = ac.generate_collation_from_tree(g, n_verses, p_new_locus, p_mod, p_inov)
    loc_sizes,seqs = ac.reduce_collation(inv,seqs)

    st  = bd.generate_stemma(g, assign_length=True)

    wits_seqs = {}
    dates = {}
    for w,s in seqs.items():
        sequence = []
        for l in s.values():
            for app in l.values():
                sequence.append(app)
        wits_seqs[f'ms_{w}'] = ''.join(map(str,sequence))
        dates[f'ms_{w}'] = int(st.nodes[w]['birth_time'])

    max_val = max([max(l.values()) for l in loc_sizes.values() if l != {}])

    nexus_sequence = generate_nexus(wits_seqs, dates, 'test', max_val)
    newick_graph = to_newick(st, root=bd.root(st), node_attrs=['state'], edge_attrs=['length'])

    with open(f'{path}/sequences_{k}.nex', 'w') as f:
        f.write(nexus_sequence)
    
    with open(f'{path}/tree_{k}.newick', 'w') as f:
        f.write(newick_graph)

  0%|          | 0/100 [00:00<?, ?it/s]